# Open Duck Mini V2｜自由なお盆＋自由な湯飲み

140M学習済みONNXを再利用し、**再学習せず**に物理接触と落下を検証します。お盆は `freejoint`、頭部には見えない接触パッドを追加します。摩擦：頭・お盆0.3、湯飲み・お盆0.1（いずれも検証用仮値）。

In [ ]:
from pathlib import Path
import subprocess
import shutil

ROOT = Path("/content/Open_Duck_Playground")
PYTHON = ROOT / ".venv/bin/python"

# ==========================================
# 1. リポジトリを取得
# ==========================================

if not (ROOT / "playground/open_duck_mini_v2/mujoco_infer.py").is_file():
    if ROOT.exists():
        # 既存ファイルを消さずに、不足コードだけ補完
        FRESH = Path("/content/Open_Duck_Playground_fresh")

        if not (FRESH / "playground").is_dir():
            subprocess.run(
                [
                    "git", "clone",
                    "https://github.com/apirrone/Open_Duck_Playground.git",
                    str(FRESH)
                ],
                check=True,
            )

        for src in (FRESH / "playground").rglob("*"):
            if not src.is_file() or "__pycache__" in src.parts:
                continue

            dst = ROOT / src.relative_to(FRESH)
            if not dst.exists():
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src, dst)
    else:
        subprocess.run(
            [
                "git", "clone",
                "https://github.com/apirrone/Open_Duck_Playground.git",
                str(ROOT)
            ],
            check=True,
        )

# ==========================================
# 2. uvをインストール
# ==========================================

subprocess.run(
    ["python", "-m", "pip", "-q", "install", "uv"],
    check=True,
)

# ==========================================
# 3. 仮想環境を作成
# ==========================================

if not PYTHON.is_file():
    subprocess.run(
        ["uv", "venv", str(ROOT / ".venv")],
        check=True,
    )

# ==========================================
# 4. 必要なライブラリをインストール
# ==========================================

packages = [
    "mujoco",
    "mujoco-mjx",
    "playground",
    "onnxruntime",
    "imageio[ffmpeg]",
    "numpy",
    "jax[cpu]",
    "ml-collections",
]

subprocess.run(
    [
        "uv", "pip", "install",
        "--python", str(PYTHON),
        *packages,
    ],
    check=True,
)

# ==========================================
# 5. 動作確認
# ==========================================

check = """
import mujoco
import onnxruntime
import imageio
import playground.open_duck_mini_v2.mujoco_infer

print("Environment OK")
print("MuJoCo:", mujoco.__version__)
"""

result = subprocess.run(
    [str(PYTHON), "-c", check],
    cwd=ROOT,
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.stderr:
    print(result.stderr)

result.check_returncode()

print("Ready:", ROOT)

Environment OK
MuJoCo: 3.16.0

Ready: /content/Open_Duck_Playground


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import shutil
ROOT=Path('/content/Open_Duck_Playground')
XML_DIR=ROOT/'playground/open_duck_mini_v2/xmls'
DRIVE=Path('/content/drive/MyDrive/OpenDuckRL')
MODEL=ROOT/'walk_140m.onnx'
if not ROOT.is_dir() or not (ROOT/'.venv/bin/python').is_file():
    raise RuntimeError('既存ノートブックの環境構築セル（.venv作成と依存関係インストール）を先に実行してください')
if not (XML_DIR/'scene_delivery_tea_v2.xml').is_file():
    src=DRIVE/'delivery_140m/source/scene_delivery_tea_v2.xml'
    XML_DIR.mkdir(parents=True,exist_ok=True)
    shutil.copy2(src,XML_DIR/src.name)
if not MODEL.is_file():
    shutil.copy2(DRIVE/'soccer_140m/model/walk_140m.onnx',MODEL)
print('Ready:', MODEL, XML_DIR/'scene_delivery_tea_v2.xml')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ready: /content/Open_Duck_Playground/walk_140m.onnx /content/Open_Duck_Playground/playground/open_duck_mini_v2/xmls/scene_delivery_tea_v2.xml


## 1. XML生成（頭に固定しないお盆）

既存の `scene_delivery_tea_v2.xml` から生成します。元XMLのキーフレームが23要素でも、MuJoCoの補完済み値から再構築します。

In [ ]:
from pathlib import Path
import subprocess
import re

ROOT = Path("/content/Open_Duck_Playground")
PYTHON = ROOT / ".venv/bin/python"
SCRIPT = ROOT / "create_delivery_free_tray.py"

if not SCRIPT.is_file():
    raise FileNotFoundError(SCRIPT)

code = SCRIPT.read_text(encoding="utf-8")

new_function = '''
def box_extent_z(model, data, geom_id):
    """ジオメトリのワールドZ方向の半幅を求める。"""
    geom_type = model.geom_type[geom_id]
    rot = data.geom_xmat[geom_id].reshape(3, 3)
    size = model.geom_size[geom_id]

    if geom_type == mujoco.mjtGeom.mjGEOM_BOX:
        return float(np.dot(np.abs(rot[2]), size[:3]))

    if geom_type == mujoco.mjtGeom.mjGEOM_CYLINDER:
        radius, half_height = size[:2]
        return float(
            radius * np.linalg.norm(rot[2, :2])
            + half_height * abs(rot[2, 2])
        )

    if geom_type == mujoco.mjtGeom.mjGEOM_SPHERE:
        return float(size[0])

    if geom_type == mujoco.mjtGeom.mjGEOM_ELLIPSOID:
        return float(np.linalg.norm(rot[2] * size[:3]))

    if geom_type == mujoco.mjtGeom.mjGEOM_CAPSULE:
        return float(
            size[0] + size[1] * abs(rot[2, 2])
        )

    raise ValueError(
        f"Unsupported geom type: "
        f"{model.geom(geom_id).name} ({geom_type})"
    )
'''

# def box_extent_z から次のトップレベル関数までを置換
pattern = r"(?ms)^def box_extent_z\(.*?(?=^def |^if __name__|\\Z)"

match = re.search(pattern, code)

if not match:
    raise RuntimeError(
        "box_extent_z関数が見つかりません。"
        "スクリプトを確認してください。"
    )

# 元のファイルをバックアップ
backup = SCRIPT.with_suffix(".py.bak")
if not backup.exists():
    backup.write_text(code, encoding="utf-8")

patched = code[:match.start()] + new_function.strip() + "\n\n" + code[match.end():]

SCRIPT.write_text(patched, encoding="utf-8")

print("Patched:", SCRIPT)
print("Backup:", backup)

# 修正後の構文を確認
subprocess.run(
    [str(PYTHON), "-m", "py_compile", str(SCRIPT)],
    check=True,
)

print("Syntax OK")

Patched: /content/Open_Duck_Playground/create_delivery_free_tray.py
Backup: /content/Open_Duck_Playground/create_delivery_free_tray.py.bak
Syntax OK


In [ ]:
from pathlib import Path
import subprocess

ROOT = Path("/content/Open_Duck_Playground")
PYTHON = ROOT / ".venv/bin/python"
SCRIPT = ROOT / "create_delivery_free_tray.py"

if not SCRIPT.is_file():
    raise FileNotFoundError(SCRIPT)

code = SCRIPT.read_text(encoding="utf-8")

# 修正対象は2箇所のみ
replacements = {
    "int(interim.joint('yunomi_freejoint').qposadr)":
    "int(interim.jnt_qposadr[interim.joint('yunomi_freejoint').id])",

    "int(interim.joint('delivery_tray_freejoint').qposadr)":
    "int(interim.jnt_qposadr[interim.joint('delivery_tray_freejoint').id])",
}

for old, new in replacements.items():
    if old in code:
        code = code.replace(old, new)
        print("Patched:", old)
    elif new in code:
        print("Already patched:", new)
    else:
        raise RuntimeError(
            f"必要な関節アドレス取得処理が見つかりません: {old}"
        )

# ファイルに保存
SCRIPT.write_text(code, encoding="utf-8")

# 構文チェック
subprocess.run(
    [str(PYTHON), "-m", "py_compile", str(SCRIPT)],
    check=True,
)

print("\nSyntax OK")

# XMLを生成
result = subprocess.run(
    [str(PYTHON), str(SCRIPT)],
    cwd=ROOT,
    text=True,
    capture_output=True,
)

print("\n=== STDOUT ===")
print(result.stdout)

if result.stderr:
    print("=== STDERR ===")
    print(result.stderr)

result.check_returncode()

# 生成されたXMLを確認
xml_path = (
    ROOT
    / "playground/open_duck_mini_v2/xmls"
    / "scene_delivery_free_tray.xml"
)

if not xml_path.is_file():
    raise FileNotFoundError(xml_path)

print("\nXML generation completed!")

Already patched: int(interim.jnt_qposadr[interim.joint('yunomi_freejoint').id])
Patched: int(interim.joint('delivery_tray_freejoint').qposadr)

Syntax OK

=== STDOUT ===
Created: /content/Open_Duck_Playground/playground/open_duck_mini_v2/xmls/scene_delivery_free_tray.xml
Actuators=14  nq=37 nv=34 keys=1
Tray freejoint: delivery_tray_freejoint
Cup freejoint: yunomi_freejoint
Tray normal: [0. 0. 1.]
Head pad top Z=0.3739 tray bottom Z=0.3744 gap=0.0005 m
Tray/cup friction: 0.1 head/tray friction: 0.3


XML generation completed!


In [ ]:
from pathlib import Path
import subprocess

ROOT = Path("/content/Open_Duck_Playground")
PYTHON = ROOT / ".venv/bin/python"
SCRIPT = ROOT / "create_delivery_free_tray.py"

code = SCRIPT.read_text(encoding="utf-8")

old = "PAD_FRICTION = 0.30"
new = "PAD_FRICTION = 0.20"

if old in code:
    code = code.replace(old, new)
    SCRIPT.write_text(code, encoding="utf-8")
    print("摩擦係数を 0.30 → 0.20 に変更しました")
elif new in code:
    print("すでに摩擦係数は0.20です")
else:
    raise RuntimeError("PAD_FRICTIONの設定が見つかりません")

# XMLを再生成
subprocess.run(
    [str(PYTHON), str(SCRIPT)],
    cwd=ROOT,
    check=True,
)

print("\n設定完了")
print("頭部・お盆: 0.20")
print("お盆・湯飲み: 0.10")

摩擦係数を 0.30 → 0.20 に変更しました

設定完了
頭部・お盆: 0.20
お盆・湯飲み: 0.10


## 2. 140Mモデルで歩行評価・録画

同じONNXで検証します。まずは落下するかを観察してください。

In [ ]:
from pathlib import Path
import subprocess,os

ROOT=Path('/content/Open_Duck_Playground')
source_candidates=[ROOT/'record_delivery_tea_v2.py', ROOT/'record_delivery_tea_slide.py', Path('/content/drive/MyDrive/OpenDuckRL/delivery_140m/source/record_delivery_tea_v2.py')]
src=next((p for p in source_candidates if p.is_file()),None)

if src is None: raise FileNotFoundError('record_delivery_tea_v2.py が見つかりません')
code=src.read_text(encoding='utf-8')
old='scene_delivery_tea_v2.xml'

if old not in code: raise RuntimeError('録画元のXML指定を確認してください')
out_script=ROOT/'record_delivery_free_tray.py'
out_script.write_text(code.replace(old,'scene_delivery_free_tray.xml'),encoding='utf-8')
VIDEO=Path('/content/delivery_free_tray.mp4')
result=subprocess.run([str(ROOT/'.venv/bin/python'),str(out_script),'--model',str(ROOT/'walk_140m.onnx'),'--output',str(VIDEO),'--seconds','10'],cwd=ROOT,env={**os.environ,'MUJOCO_GL':'egl'},text=True,capture_output=True)

print(result.stdout);print(result.stderr);result.check_returncode()

from IPython.display import Video,display

display(Video(str(VIDEO),embed=True,width=800))

playground/open_duck_mini_v2/xmls/scene_delivery_free_tray.xml
[Poly ref data] Processing ...
[Poly ref data] Done processing
joint names: ['floating_base', 'left_hip_yaw', 'left_hip_roll', 'left_hip_pitch', 'left_knee', 'left_ankle', 'neck_pitch', 'head_pitch', 'head_yaw', 'head_roll', 'right_hip_yaw', 'right_hip_roll', 'right_hip_pitch', 'right_knee', 'right_ankle', 'yunomi_freejoint', 'tea_slosh_x', 'tea_slosh_y', 'delivery_tray_freejoint']
actuator names: ['left_hip_yaw', 'left_hip_roll', 'left_hip_pitch', 'left_knee', 'left_ankle', 'neck_pitch', 'head_pitch', 'head_yaw', 'head_roll', 'right_hip_yaw', 'right_hip_roll', 'right_hip_pitch', 'right_knee', 'right_ankle']
backlash joint names: ['yunomi_freejoint', 'tea_slosh_x', 'tea_slosh_y', 'delivery_tray_freejoint']
Start XY: [0. 0.]
End XY: [ 0.63907443 -0.12296592]
Delta X: 0.6391 m
Delta Y: -0.1230 m
Displacement: 0.6508 m
Saved: /content/delivery_free_tray.mp4




## 3. Driveへ成果物を保存

In [ ]:
from pathlib import Path
import shutil
ROOT=Path('/content/Open_Duck_Playground')
DEST=Path('/content/drive/MyDrive/OpenDuckRL/delivery_140m/free_tray_baseline')
items={'video/delivery_free_tray.mp4':Path('/content/delivery_free_tray.mp4'),'source/create_delivery_free_tray.py':ROOT/'create_delivery_free_tray.py','source/record_delivery_free_tray.py':ROOT/'record_delivery_free_tray.py','source/scene_delivery_free_tray.xml':ROOT/'playground/open_duck_mini_v2/xmls/scene_delivery_free_tray.xml','model/walk_140m.onnx':ROOT/'walk_140m.onnx'}
missing=[str(p) for p in items.values() if not p.is_file()]
if missing: raise FileNotFoundError('Missing:\n'+'\n'.join(missing))
for rel,src in items.items():
    dst=DEST/rel;dst.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(src,dst);print('Saved',dst)
print('Complete:',DEST)

Saved /content/drive/MyDrive/OpenDuckRL/delivery_140m/free_tray_baseline/video/delivery_free_tray.mp4
Saved /content/drive/MyDrive/OpenDuckRL/delivery_140m/free_tray_baseline/source/create_delivery_free_tray.py
Saved /content/drive/MyDrive/OpenDuckRL/delivery_140m/free_tray_baseline/source/record_delivery_free_tray.py
Saved /content/drive/MyDrive/OpenDuckRL/delivery_140m/free_tray_baseline/source/scene_delivery_free_tray.xml
Saved /content/drive/MyDrive/OpenDuckRL/delivery_140m/free_tray_baseline/model/walk_140m.onnx
Complete: /content/drive/MyDrive/OpenDuckRL/delivery_140m/free_tray_baseline
